# Trace Count v0: One-Click Colab Run

Run all cells from top to bottom. By default this notebook runs a complete **debug-size** experiment: data generation, all loss-mask regimes, evaluation, summary tables, hidden-state probes, and plots. Switch `RUN_FULL_V0 = True` only after the debug run works.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
DEFAULT_REPO_DIR = Path("/content/Synthetic_CoT_NiaH_Count") if Path("/content").exists() else Path.cwd() / "Synthetic_CoT_NiaH_Count"

if Path("pyproject.toml").exists() and Path("src/trace_counting").exists():
    repo_dir = Path.cwd()
else:
    repo_dir = DEFAULT_REPO_DIR
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

if (Path.cwd() / ".git").exists():
    subprocess.run(["git", "pull", "--ff-only"], check=False)

print("Repo:", Path.cwd())
print("Python:", sys.version.split()[0])

## Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

## Runtime Settings

The default run is intentionally small but complete: it runs every loss-mask regime and displays the final comparison table. Increase `MAX_STEPS` for more meaningful accuracy.

In [ ]:
RUN_FULL_V0 = False
RUN_TESTS = True

if RUN_FULL_V0:
    DATA_CONFIG = "configs/experiment/v0.yaml"
    DATA_DIR = "data/trace_count_v0"
    OUT_ROOT = "runs/trace_count_v0"
    MODEL_CONFIG = "configs/model/small_main.yaml"
    MODEL_NAME = "small_main"
    SWEEP_SEEDS = "0,1,2"
    MAX_STEPS = 50000
    BATCH_SIZE = 128
    EVAL_LIMIT = None
    PROBE_LIMIT = 4096
else:
    DATA_CONFIG = "configs/experiment/debug.yaml"
    DATA_DIR = "data/trace_count_v0_debug"
    OUT_ROOT = "runs/trace_count_v0_debug_sweep"
    MODEL_CONFIG = "configs/model/tiny_debug.yaml"
    MODEL_NAME = "tiny_debug"
    SWEEP_SEEDS = "0"
    MAX_STEPS = 100
    BATCH_SIZE = 32
    EVAL_LIMIT = 128
    PROBE_LIMIT = 128

MAIN_RUN = f"{OUT_ROOT}/{MODEL_NAME}/completion_final_weighted_fw10_seed0"
SUMMARY_CSV = f"{OUT_ROOT}/summary.csv"

print({
    "RUN_FULL_V0": RUN_FULL_V0,
    "DATA_DIR": DATA_DIR,
    "OUT_ROOT": OUT_ROOT,
    "MODEL_CONFIG": MODEL_CONFIG,
    "MAX_STEPS": MAX_STEPS,
    "MAIN_RUN": MAIN_RUN,
})

## Tests

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Skipping tests")

## Generate Data

In [ ]:
subprocess.run([
    sys.executable,
    "scripts/run_pipeline.py",
    "--config", DATA_CONFIG,
    "--stage", "data",
], check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

with open(Path(DATA_DIR) / "dataset_metadata.json") as f:
    dataset_metadata = json.load(f)
display(pd.DataFrame(dataset_metadata["split_counts"].items(), columns=["split", "examples"]))

sample = pd.read_json(Path(DATA_DIR) / "train.jsonl", lines=True, nrows=3)
display(sample[["example_id", "seq_len", "count", "full_tokens"]])

## Train + Evaluate All Loss-Mask Regimes

In [ ]:
cmd = [
    sys.executable,
    "scripts/run_loss_mask_sweep.py",
    "--data_dir", DATA_DIR,
    "--model_config", MODEL_CONFIG,
    "--model_name", MODEL_NAME,
    "--out_root", OUT_ROOT,
    "--seeds", SWEEP_SEEDS,
    "--max_steps", str(MAX_STEPS),
    "--batch_size", str(BATCH_SIZE),
]
if EVAL_LIMIT is not None:
    cmd += ["--eval_limit", str(EVAL_LIMIT)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Summary Table

In [ ]:
subprocess.run([
    sys.executable,
    "-m", "trace_counting.summarize",
    "--runs_dir", OUT_ROOT,
    "--out_csv", SUMMARY_CSV,
    "--print_markdown",
], check=True)

summary = pd.read_csv(SUMMARY_CSV)
display(summary.sort_values(["split", "tf_count_acc", "ar_count_acc"], ascending=[True, False, False]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = summary.melt(
    id_vars=["split", "loss_mask", "final_weight"],
    value_vars=["tf_count_acc", "ar_count_acc", "trace_exact", "format_valid"],
    var_name="metric",
    value_name="value",
)
plot_df["regime"] = plot_df["loss_mask"] + plot_df["final_weight"].fillna("").astype(str).map(lambda x: "" if x == "" else " fw=" + x)

g = sns.catplot(
    data=plot_df,
    x="regime",
    y="value",
    hue="metric",
    col="split",
    kind="bar",
    col_wrap=2,
    height=4,
    aspect=1.6,
    sharey=True,
)
for ax in g.axes.flatten():
    ax.tick_params(axis="x", rotation=75)
    ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

## Detailed Metrics For The Main Candidate

In [ ]:
main_run = Path(MAIN_RUN)
print("Main run:", main_run)

train_log = pd.read_json(main_run / "train_log.jsonl", lines=True)
display(train_log.tail())

for metric_path in sorted((main_run / "eval").glob("*_metrics.json")):
    if metric_path.name == "summary_metrics.json":
        continue
    with open(metric_path) as f:
        metrics = json.load(f)
    split = metrics["split"]
    tf = metrics.get("teacher_forced", {})
    ar = metrics.get("autoregressive", {})
    print(f"\n{split}")
    print("teacher_forced:", {k: tf.get(k) for k in ["count_accuracy", "mean_absolute_error", "undercount_rate", "overcount_rate", "count_nll"]})
    print("autoregressive:", {k: ar.get(k) for k in ["count_accuracy", "mean_absolute_error", "trace_exact_match", "format_validity", "invalid_answer_rate"]})

In [ ]:
pred_path = main_run / "eval" / "predictions_val_id.jsonl"
predictions = pd.read_json(pred_path, lines=True)
cols = [
    "example_id", "seq_len", "true_count", "tf_pred_count", "tf_correct",
    "ar_pred_count", "ar_format_valid", "ar_trace_exact_match", "generated_tokens",
]
display(predictions[[c for c in cols if c in predictions.columns]].head(20))

## Hidden-State Probes And Saved Plots

In [ ]:
subprocess.run([
    sys.executable,
    "-m", "trace_counting.probes",
    "--checkpoint", str(main_run / "checkpoints" / "final"),
    "--data_dir", DATA_DIR,
    "--split", "val_id",
    "--out_dir", str(main_run / "probes"),
    "--anchors", "ans,think_open,think_close,source_marker,trace_index,trace_marker",
    "--layers", "all",
    "--limit", str(PROBE_LIMIT),
], check=True)

subprocess.run([
    sys.executable,
    "-m", "trace_counting.plots",
    "--run_dir", str(main_run),
], check=True)

In [ ]:
probe_summary = pd.read_csv(main_run / "probes" / "probe_summary.csv")
display(probe_summary.sort_values(["target", "probe_type", "anchor", "layer"]).head(50))

In [ ]:
from IPython.display import Image, display

for path in sorted((main_run / "plots").glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

## Full v0 Run

After the default notebook run succeeds, set `RUN_FULL_V0 = True` near the top and run all cells again. The full setting uses `small_main`, three seeds, and 50k steps per regime, so it is much slower than the debug sweep.